# Greedy z=w sweep — finish the 640-solved tail (r1 / r2 @ 500k)

Runs the **remaining** heavy idx of one `z=w` arm over the 640 solved MS(1190) presentations at a
500k node budget, and lands crash-safe resumable JSONL in Drive. Run **one arm per Colab runtime**
(open two copies: set `ARM="r1"` in one, `ARM="r2"` in the other).

**Why this notebook and not the driver's own light/heavy split:** the 12k label file the driver uses
to classify light-vs-heavy idx was overwritten by the website bundle rebuild, so its classifier is
now stale. This notebook sidesteps it by running the light and heavy phases at the **same** worker
count — every remaining idx is memory-heavy, so one safe count is correct for all of them.

**Resume:** it seeds the committed partial results (595 done for r1, 579 for r2) into the Drive
output dir, so the run skips everything already finished and only computes the tail (45 for r1, 61
for r2). Safe to re-run after any Colab disconnect — it picks up where it stopped.

**Run order:** `s1 CONFIG` → `s2 SETUP` → `s3 RUN` → (`s4 TALLY` anytime to check progress).

### Worker count by Colab tier (each heavy n=3 @500k peaks ~5–8 GB RSS)

| RAM | tier | safe workers |
|----|------|----|
| ~13 GB | free | **1** |
| ~25 GB | Pro (standard) | **2** |
| ~51 GB | Pro high-RAM | **5** |

Leave `WORKERS=None` to auto-pick from detected RAM (≈ RAM_GB // 9), or pin an int.

In [ ]:
# ==================== s1  CONFIG  (edit ONLY this cell) ====================
ARM           = "r1"          # <-- "r1" in one Colab, "r2" in the other
BUDGET        = 500_000        # node budget per presentation (matched-budget comparison)
WORKERS       = None           # None = auto from RAM (~RAM_GB//9); or pin: free->1, Pro->2, high-RAM->5

REPO_URL      = "https://github.com/Avi161/ACSolverX.git"
BRANCH        = "test/stable-ac-moves"
REPO_DIR      = "ACSolverX"
UPDATE_REPO   = True           # git reset --hard origin/BRANCH on restart so a rerun picks up latest push
MOUNT_DRIVE   = True
SEED_FROM_REPO= True           # copy committed partial results into Drive so resume skips finished idx

START, END, MAX_LEN = 0, 640, 24
print(f"config: ARM={ARM}  BUDGET={BUDGET:,}  WORKERS={WORKERS}")

In [ ]:
# ==================== s2  SETUP  (clone / pull / install / mount / seed) ====================
import os, sys, shutil, subprocess
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")   # confirm the commit has run_greedy_sweep.py + results/solved640
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while REPO_ROOT != "/" and not os.path.isdir(os.path.join(REPO_ROOT, "envs")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# machine profile -> worker recommendation
try:
    kb = int(next(l for l in open("/proc/meminfo") if l.startswith("MemTotal")).split()[1])
    TOTAL_GB = kb / 1024 / 1024
except Exception:
    TOTAL_GB = float("nan")
CPU = os.cpu_count() or 2
AUTO_W = max(1, min(CPU - 1, int(TOTAL_GB // 9))) if TOTAL_GB == TOTAL_GB else 1
print(f"repo: {REPO_ROOT}")
print(f"RAM: {TOTAL_GB:.1f} GB   CPU: {CPU}   auto workers -> {AUTO_W}")

# output dir (arm-specific so two runtimes never write the same Drive files)
if IN_COLAB and MOUNT_DRIVE:
    OUT_DIR = f"/content/drive/MyDrive/solved640_{ARM}"
else:
    OUT_DIR = os.path.join(REPO_ROOT, "results", "solved640")

# seed committed partials so done_idx skips finished idx (only the tail recomputes)
if SEED_FROM_REPO:
    src_root = os.path.join(REPO_ROOT, "results", "solved640")
    for sub, fn in (("solved", f"calibration_{ARM}.jsonl"), ("paths", f"paths_{ARM}.jsonl")):
        src, dst = os.path.join(src_root, sub, fn), os.path.join(OUT_DIR, sub, fn)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst); print(f"seeded {fn} -> {dst}")
        elif os.path.exists(dst):
            print(f"exists (resuming from Drive): {dst}")
print("out_dir:", OUT_DIR)

In [ ]:
# ==================== s3  RUN  (streams live; resumable — re-run safely after a disconnect) ====================
W = WORKERS if WORKERS else AUTO_W
script = os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator", "run_greedy_sweep.py")
cmd = [sys.executable, script,
       "--arms", ARM, "--budget", str(BUDGET),
       "--start", str(START), "--end", str(END), "--max_len", str(MAX_LEN),
       "--light_workers", str(W), "--heavy_workers", str(W),   # EQUAL -> classification-independent, OOM-safe
       "--out_dir", OUT_DIR]
print(f"arm={ARM}  workers={W}  budget={BUDGET:,}")
print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)
print(f"\n-> {OUT_DIR}  (solved/calibration_{ARM}.jsonl  +  paths/paths_{ARM}.jsonl)")

In [ ]:
# ==================== s4  TALLY  (run anytime to check progress) ====================
import json
cf = os.path.join(OUT_DIR, "solved", f"calibration_{ARM}.jsonl")
done = solved = 0
if os.path.exists(cf):
    for line in open(cf):
        line = line.strip()
        if not line:
            continue
        try:
            r = json.loads(line)
        except Exception:
            continue
        if r.get("dataset") == "1190MS" and r.get("arm") == ARM and r.get("budget_nodes") == BUDGET:
            done += 1; solved += int(bool(r.get("solved")))
print(f"{ARM}: done={done}/640  solved={solved}  exhausted={done-solved}  remaining={640-done}")
print("When done=640, copy this Drive folder back and paste to me; I'll fold it into results/solved640 + rebuild the site.")